# In Class Activity April 14th, 2026

In [1]:
%%capture
pip install optuna

### Importing libraries, preparing data, initial EDA

In [2]:
# importing libraries
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import optuna


c:\Users\donya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [28]:
# importing data
adult = pd.read_csv('C:\\Users\\donya\\Documents\\GSB 545\\GSB-545\\In Class Assignments\\adult.csv')
adult.head(20)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,?,227026,HS-grad,9,Never-married,?,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


In [5]:
adult.dtypes

age                int64
workclass            str
fnlwgt             int64
education            str
educational-num    int64
marital-status       str
occupation           str
relationship         str
race                 str
gender               str
capital-gain       int64
capital-loss       int64
hours-per-week     int64
native-country       str
income               str
dtype: object

In [6]:
# initial EDA with sweetviz
report = sv.analyze(adult)
report.show_html('sweet_report.html')


Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)


Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### In the markdown cell below describe what you learned from your EDA and how it will inform your modeling decisions





- The outcome variable (income >50K or <=50K>) is imbalanced. There are more individuals in this data with incomes <=50K. I will have to take this into account when defining my models (there are some models that address class imbalance), use stratified K fold cross validation, and make sure my outcome variable is stratified in my train test split.
- There are a handful of variables with missing data (workclass, native-country) marked with "?". This isn't an issue for XGBoost though, as this model can natively handle missing values.
- The average individual in this dataset is a ~37 years old, working full time in the private sector, a high school graduate, married, male, white, and from the United States.
- There are 6 quantitative variables and 9 categorical variables, including the outcome variable, income. We do not need to scale our variables for XGBoost. XGBoost can handle categorical variables natively if we transform our categorical variales to the category data type and set `enable_categorical=True`.
- There is a very strong, positive (potentially = 1) relationship between education and educational_num. One of these variables should be removed before modeling to avoid using redundant predictors in the models.

### Data Preprocessing and Baseline Model

In [29]:
# data cleaning and preprocessing

# changing ? to NaN
adult = adult.replace('?', np.nan)

#education and education num are redundant, so we can drop one of them
adult = adult.drop('education', axis=1)

# target variable is income with 2 levels, so we can encode it as 0 and 1
adult['income'] = adult['income'].apply(lambda x: 1 if x == '>50K' else 0)

# make 'Married-spouse-absent', 'Married-AF-spouse', 'Married-civ-spouse' one category
adult['marital-status'] = adult['marital-status'].replace({
    "Married-spouse-absent": "Married",
    "Married-AF-spouse": "Married",
    "Married-civ-spouse": "Married"
})

# change dtype categorical variables to category
categorical_cols = adult.select_dtypes(include='object').columns
adult[categorical_cols] = adult[categorical_cols].astype('category')

adult.head(20)

C:\Users\donya\AppData\Local\Temp\ipykernel_1416\321097776.py:20: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = adult.select_dtypes(include='object').columns


,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,9,Married,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,12,Married,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,10,Married,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
5,34,Private,198693,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,0
6,29,NaN,227026,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,0
7,63,Self-emp-not-inc,104626,15,Married,Prof-specialty,Husband,White,Male,3103,0,32,United-States,1
8,24,Private,369667,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,0
9,55,Private,104996,4,Married,Craft-repair,Husband,White,Male,0,0,10,United-States,0


In [30]:
# defining features and target variable
X = adult.drop('income', axis=1)
y = adult['income']

# splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, 
                                                    random_state=321, stratify=y)

In [12]:
X.head()

,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States
1,38,Private,89814,9,Married,Farming-fishing,Husband,White,Male,0,0,50,United-States
2,28,Local-gov,336951,12,Married,Protective-serv,Husband,White,Male,0,0,40,United-States
3,44,Private,160323,10,Married,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States


In [13]:
y.head()

0    0
1    0
2    1
3    1
4    0
Name: income, dtype: int64

In [17]:
# building xgboost default model and evaluating with stratified k-fold cross validation
xgb_cv = XGBClassifier(enable_categorical=True, random_state=321)

#stratifies target across each fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=321)

cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean()}') 
print(f'Std Deviation of F1 score: {cv_scores.std()}')

#fit default XGBoost model and evaluate on test set
xgb_cv.fit(X_train, y_train)
y_pred_default = xgb_cv.predict(X_test)
print("Default Model F1 Score:", f1_score(y_test, y_pred_default))
print("Classification Report:")
print(classification_report(y_test, y_pred_default))

Cross-validated F1 scores: [0.71584319 0.71983356 0.69546941 0.69953917 0.70159027]
Mean F1 score: 0.7064551209872654
Std Deviation of F1 score: 0.009584379733532048
Default Model F1 Score: 0.7059639389736477
Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.94      0.92      7431
           1       0.77      0.65      0.71      2338

    accuracy                           0.87      9769
   macro avg       0.83      0.80      0.81      9769
weighted avg       0.87      0.87      0.87      9769



### Use the markdown cell below to describe your baseline model performance and how you will try to improve it

For a baseline model, the performance is decent. The average F1 score is 0.706, indicating reasonably strong performance in identifying individuals with incomes greater than 50K without making too many incorrect predictions of such individuals. However, this model can definitely be improved, as seen in the classification report. The model is performing well with identifying individuals whose incomes are <= 50K, but struggling with classifying the minority class (individuals whose incomes are > 50K). I would like to tune XGBoost features like learning rate and min_child_weight to control overfitting, max depth to control tree complexity, and scale_pos_weight to ensure that the XGBoost model makes less incorrect classifications of the income > 50K class.

### Model feature exploration

In the code cell below, explore different features of XGBoost and how they work (e.g. scale_pos_weight, max_depth, learning_rate).
Use stratified k-fold cross or repeated stratified k-fold cross validation with your model building. 
You should explore at least 3 different features of XGBoost.
Identify the model that performs best.

In [15]:
#XGBoost model with small learning rate
xgb_m1 = XGBClassifier(enable_categorical=True, random_state=321, learning_rate = 0.01)
cv_scores_m1 = cross_val_score(xgb_m1, X, y, cv=skf, scoring='f1')

#XGBoost model with large learning rate
xgb_m2 = XGBClassifier(enable_categorical=True, random_state=321, learning_rate = 0.3)
cv_scores_m2 = cross_val_score(xgb_m2, X, y, cv=skf, scoring='f1')

#XGBoost model with less complexity
xgb_m3 = XGBClassifier(enable_categorical=True, random_state=321, max_depth = 2)
cv_scores_m3 = cross_val_score(xgb_m3, X, y, cv=skf, scoring='f1')

#XGBoost model with more complexity
xgb_m4 = XGBClassifier(enable_categorical=True, random_state=321, max_depth = 8)
cv_scores_m4 = cross_val_score(xgb_m4, X, y, cv=skf, scoring='f1')

#XGBoost model with small min_child_weight
xgb_m5 = XGBClassifier(enable_categorical=True, random_state=321, min_child_weight = 2)
cv_scores_m5 = cross_val_score(xgb_m5, X, y, cv=skf, scoring='f1')

#XGBoost model with large min_child_weight
xgb_m6 = XGBClassifier(enable_categorical=True, random_state=321, min_child_weight = 10)
cv_scores_m6 = cross_val_score(xgb_m6, X, y, cv=skf, scoring='f1')

#XGBoost model with less weight on class 1
xgb_m7 = XGBClassifier(enable_categorical=True, random_state=321, scale_pos_weight = 3)
cv_scores_m7 = cross_val_score(xgb_m7, X, y, cv=skf, scoring='f1')

#XGBoost model with more weight on class 1
xgb_m8 = XGBClassifier(enable_categorical=True, random_state=321, scale_pos_weight = 10)
cv_scores_m8 = cross_val_score(xgb_m8, X, y, cv=skf, scoring='f1')

#model with all the best individual features
xgb_m9 = XGBClassifier(enable_categorical=True, random_state=321, learning_rate = 0.3, max_depth = 2, min_child_weight = 2, scale_pos_weight = 3)
cv_scores_m9 = cross_val_score(xgb_m9, X, y, cv=skf, scoring='f1')


print("##### Model 1 #####")
print(f'Cross-validated F1 scores: {cv_scores_m1}')
print(f'Mean F1 score: {cv_scores_m1.mean()}') 
print(f'Std Deviation of F1 score: {cv_scores_m1.std()}\n')

print("##### Model 2 #####")
print(f'Cross-validated F1 scores: {cv_scores_m2}')
print(f'Mean F1 score: {cv_scores_m2.mean()}') 
print(f'Std Deviation of F1 score: {cv_scores_m2.std()}\n')

print("##### Model 3 #####")
print(f'Cross-validated F1 scores: {cv_scores_m3}')
print(f'Mean F1 score: {cv_scores_m3.mean()}') 
print(f'Std Deviation of F1 score: {cv_scores_m3.std()}\n')

print("##### Model 4 #####")
print(f'Cross-validated F1 scores: {cv_scores_m4}')
print(f'Mean F1 score: {cv_scores_m4.mean()}') 
print(f'Std Deviation of F1 score: {cv_scores_m4.std()}\n')

print("##### Model 5 #####")
print(f'Cross-validated F1 scores: {cv_scores_m5}')
print(f'Mean F1 score: {cv_scores_m5.mean()}') 
print(f'Std Deviation of F1 score: {cv_scores_m5.std()}\n')

print("##### Model 6 #####")
print(f'Cross-validated F1 scores: {cv_scores_m6}')
print(f'Mean F1 score: {cv_scores_m6.mean()}') 
print(f'Std Deviation of F1 score: {cv_scores_m6.std()}\n')

print("##### Model 7 #####")
print(f'Cross-validated F1 scores: {cv_scores_m7}')
print(f'Mean F1 score: {cv_scores_m7.mean()}') 
print(f'Std Deviation of F1 score: {cv_scores_m7.std()}\n')

print("##### Model 8 #####")
print(f'Cross-validated F1 scores: {cv_scores_m8}')
print(f'Mean F1 score: {cv_scores_m8.mean()}') 
print(f'Std Deviation of F1 score: {cv_scores_m8.std()}\n')

print("##### Model 9 #####")
print(f'Cross-validated F1 scores: {cv_scores_m9}')
print(f'Mean F1 score: {cv_scores_m9.mean()}') 
print(f'Std Deviation of F1 score: {cv_scores_m9.std()}\n')

##### Model 1 #####
Cross-validated F1 scores: [0.57688966 0.59630156 0.58506106 0.6060438  0.58773665]
Mean F1 score: 0.5904065481141798
Std Deviation of F1 score: 0.009976811099129654

##### Model 2 #####
Cross-validated F1 scores: [0.71584319 0.71983356 0.69546941 0.69953917 0.70159027]
Mean F1 score: 0.7064551209872654
Std Deviation of F1 score: 0.009584379733532048

##### Model 3 #####
Cross-validated F1 scores: [0.71040831 0.71260028 0.705549   0.69507888 0.69280421]
Mean F1 score: 0.7032881348181748
Std Deviation of F1 score: 0.007997863143112462

##### Model 4 #####
Cross-validated F1 scores: [0.70954357 0.71013825 0.69442517 0.69088385 0.70189202]
Mean F1 score: 0.7013765703405016
Std Deviation of F1 score: 0.007773695784376615

##### Model 5 #####
Cross-validated F1 scores: [0.72255397 0.72023947 0.70755597 0.69549218 0.70807164]
Mean F1 score: 0.7107826468390847
Std Deviation of F1 score: 0.00979341508736718

##### Model 6 #####
Cross-validated F1 scores: [0.7204797  0.72204

The model that performs the best is the XGBoost model with all default values except for scale_pos_weight = 3.

### Tuning with GridSearchCV

Use the code cell below to set up your parameter grid and run GridSearchCV with your preferred model from above. You should tune 4-5 hyperparameters utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [ ]:
paramGrid = {
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3], 
    'max_depth': [3, 4, 5, 6, 8],
    'min_child_weight': [1, 2, 5, 10],
    'scale_pos_weight': [1, 2, 5, 10, 20]
}

grid_search = GridSearchCV(xgb_m7, 
                      paramGrid, 
                      cv = 5, 
                      scoring = 'f1', 
                      n_jobs = -1)

xgboost_model = grid_search.fit(X_train, y_train)

#cross-validated F1 score on training data
print("Best F1 score:", grid_search.best_score_)
print("Best params:", grid_search.best_params_)

#fit final model using best hyperparameters
xgboost_best_model = grid_search.best_estimator_
xgboost_best_model.fit(X_train, y_train)

#evaluate final model performance on held out test set
y_pred = xgboost_best_model.predict(X_test)
print("Best Model F1 Score:", f1_score(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred))

Best F1 score: 0.7314503795645442
Best params: {'learning_rate': 0.1, 'max_depth': 8, 'min_child_weight': 5, 'scale_pos_weight': 2}
Best Model F1 Score: 0.7176145339652449
Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.88      0.90      7431
           1       0.67      0.78      0.72      2338

    accuracy                           0.85      9769
   macro avg       0.80      0.83      0.81      9769
weighted avg       0.86      0.85      0.86      9769



This tuned model performed better than the baseline model due to having a larger average F1 score (the baseline model had an F1 score of 0.706, while this model's score increased to 0.73). The best parameters across the grid search turned out to be learning rate = 0.3, max_depth = 4, min_child_weight = 2, and scale_pos_weight = 2. However, when predicting the outcome variable for the held out test data, the F1 score slightly decreased from 0.731 to 0.718. This is still higher than the baseline model's F1 score though. In the classification report, we can see that compared to the baseline model, the precision for class 1 went down, but recall increased. This indicates that there is a lower false negative rate, but higher false positive rate.

### Tuning with RandomizedSearchCV

Using the code cell below as a starting point, tune your preferred model from above. Tune the same 4-5 hyperparameters from above utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [20]:
# tuning xgboost classifier with RandomizedSearchCV
param_dist = {
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3], 
    'max_depth': [3, 4, 5, 6, 8],
    'min_child_weight': [1, 2, 5, 10],
    'scale_pos_weight': [1, 2, 5, 10, 20]
}

xgb_random = RandomizedSearchCV(
    estimator=xgb_m7,
    param_distributions=param_dist,
    n_iter=20,
    cv=skf,
    scoring="f1",
    n_jobs=-1,
    random_state=321
)

xgb_random.fit(X_train, y_train)

print(f'Best parameters from RandomizedSearchCV: {xgb_random.best_params_}')
print(f'Best F1 score from RandomizedSearchCV: {xgb_random.best_score_}')

Best parameters from RandomizedSearchCV: {'scale_pos_weight': 2, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.1}
Best F1 score from RandomizedSearchCV: 0.7159952597181247


In [22]:
# build your preferred model from above using best parameters from your RandomizedSearchCV
# and evaluate on the test set

xgb_random_best = XGBClassifier(random_state=321, scale_pos_weight=xgb_random.best_params_['scale_pos_weight'], 
                                max_depth=xgb_random.best_params_['max_depth'], 
                                learning_rate=xgb_random.best_params_['learning_rate'], 
                                min_child_weight=xgb_random.best_params_['min_child_weight'],
                                enable_categorical=True)
xgb_random_best.fit(X_train, y_train)
y_pred_random = xgb_random_best.predict(X_test)
print("RandomizedSearchCV-tuned Model F1 Score:", f1_score(y_test, y_pred_random))
print(f'Classification report for RandomizedSearchCV-tuned model:\n{classification_report(y_test, y_pred_random)}')


RandomizedSearchCV-tuned Model F1 Score: 0.7031802120141343
Classification report for RandomizedSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.92      0.87      0.90      7431
           1       0.65      0.77      0.70      2338

    accuracy                           0.85      9769
   macro avg       0.79      0.82      0.80      9769
weighted avg       0.86      0.85      0.85      9769



As with GridSearchCV, this model tuned with RandomizedSearchCV performed better in terms of cross-validated average F1 score compared to the baseline model. However, when applying this model to predict the test cases, the model performed worse, with an F1 score of 0.703. The baseline model had an F1 score of 0.706 when used to predict the test data. This likely indicates that there is more variation in model performance. Like the model tuned with GridSearchCV, this model has a decline in precision and increase in recall scores, but this model performs slightly worse than the GridSearchCV model (lower precision and recall).

### Tuning with Optuna

Using the code cell below as a starting point, tune your preferred model from above. You should tune the same 4-5 parameters as above using cross validation. Train a final model using the best hyperparameters and report your model performance.

In [25]:
# tuning with Optuna
def objective(trial):
    scale_pos_weight = trial.suggest_int('scale_pos_weight', 1, 20)
    max_depth = trial.suggest_int('max_depth', 3, 8)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    min_child_weight = trial.suggest_int('min_child_weight', 1, 10)
    
    xgb_optuna = XGBClassifier(random_state=321, scale_pos_weight=scale_pos_weight, 
                               max_depth=max_depth,  learning_rate=learning_rate, min_child_weight=min_child_weight, enable_categorical=True)
    
    cv_scores = cross_val_score(xgb_optuna, X, y, cv=skf, scoring='f1')
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'Best parameters from Optuna: {study.best_params}')
print(f'Best F1 score from Optuna: {study.best_value}')

[I 2026-04-15 08:53:48,034] A new study created in memory with name: no-name-5141b4da-f1b4-4530-b709-df816d249b16
Best trial: 0. Best value: 0.644456:   2%|▏         | 1/50 [00:02<02:19,  2.85s/it]

[I 2026-04-15 08:53:50,886] Trial 0 finished with value: 0.6444558989887446 and parameters: {'scale_pos_weight': 15, 'max_depth': 7, 'learning_rate': 0.1359671868550819, 'min_child_weight': 5}. Best is trial 0 with value: 0.6444558989887446.


Best trial: 1. Best value: 0.727906:   4%|▍         | 2/50 [00:05<02:15,  2.81s/it]

[I 2026-04-15 08:53:53,673] Trial 1 finished with value: 0.7279057104406013 and parameters: {'scale_pos_weight': 2, 'max_depth': 8, 'learning_rate': 0.13191025543713056, 'min_child_weight': 7}. Best is trial 1 with value: 0.7279057104406013.


Best trial: 1. Best value: 0.727906:   6%|▌         | 3/50 [00:08<02:10,  2.78s/it]

...

[I 2026-04-15 08:55:27,691] Trial 47 finished with value: 0.5984396869516428 and parameters: {'scale_pos_weight': 20, 'max_depth': 3, 'learning_rate': 0.20425199358109733, 'min_child_weight': 4}. Best is trial 25 with value: 0.730039057046589.


Best trial: 25. Best value: 0.730039:  98%|█████████▊| 49/50 [01:41<00:01,  1.67s/it]

[I 2026-04-15 08:55:29,349] Trial 48 finished with value: 0.709402034157205 and parameters: {'scale_pos_weight': 1, 'max_depth': 4, 'learning_rate': 0.1492131930546239, 'min_child_weight': 3}. Best is trial 25 with value: 0.730039057046589.


Best trial: 25. Best value: 0.730039: 100%|██████████| 50/50 [01:43<00:00,  2.06s/it]

[I 2026-04-15 08:55:31,268] Trial 49 finished with value: 0.6923665806250742 and parameters: {'scale_pos_weight': 5, 'max_depth': 5, 'learning_rate': 0.1648970834779227, 'min_child_weight': 6}. Best is trial 25 with value: 0.730039057046589.
Best parameters from Optuna: {'scale_pos_weight': 2, 'max_depth': 4, 'learning_rate': 0.20557071670752874, 'min_child_weight': 4}
Best F1 score from Optuna: 0.730039057046589


In [26]:
# build preferred model from above with best parameters from Optuna and evaluate on the test set
xgb_optuna_best = XGBClassifier(random_state=321, scale_pos_weight=study.best_params['scale_pos_weight'], 
                                  max_depth=study.best_params['max_depth'], 
                                  learning_rate=study.best_params['learning_rate'], 
                                  min_child_weight=study.best_params['min_child_weight'],
                                  enable_categorical=True)
xgb_optuna_best.fit(X_train, y_train)
y_pred_optuna = xgb_optuna_best.predict(X_test)
print("Optuna-tuned Model F1 Score:", f1_score(y_test, y_pred_optuna))
print(f'Classification report for Optuna-tuned model:\n{classification_report(y_test, y_pred_optuna)}')


Optuna-tuned Model F1 Score: 0.7187316500293599
Classification report for Optuna-tuned model:
              precision    recall  f1-score   support

           0       0.93      0.87      0.90      7431
           1       0.66      0.79      0.72      2338

    accuracy                           0.85      9769
   macro avg       0.80      0.83      0.81      9769
weighted avg       0.86      0.85      0.86      9769



The model tuned with Optuna performed better than the baseline XGBoost model and RandomizedSearchCV model in terms of average cross-validated F1 score (0.730) and F1 score once used to predict the test data (0.719). The model performed similarly to the model tuned with GridSearchCV, but performed slightly worse on precision (0.66 vs. 0.67) and better on recall (0.79 vs. 0.78). 

### Tuning results

In the markdown cell below describe your experience tuning with the different methods. Which produced the best results? Which do you prefer?


I have used GridSearchCV the most, so I was most comfortable with this method of tuning. However, this tuning method is too computationally expensive. The tuning took ~10 minutes to run while RandomizedSearchCV and Optuna took a maximum of ~2 minutes. I do appreciate though that GridSearchCV performs an exhaustive search over the parameter grid. This ensures that every listed combination is evaluated, which makes me feel more confident in the final model chosen. In fact, this method produced the best results in terms of F1 score. RandomizedSearchCV was my least favorite tuning method. Although this method was able to output results in seconds, a major tradeoff is solution quality. I noticed that this tuning method performed worse in some areas compared to the baseline model, so I will likely not be using it again. Optuna was my favorite tuning method by far. It took a reasonable amount of time to run, saves time with pruning, and searches intelligently compared to GridSearchCV and RandomizedSearchCV. Also, a major frustration I have with GridSearchCV is not knowing how far it is into the tuning process. I really liked that Optuna shows each trial along with a progress bar. Additionally, it seems like while Optuna takes less time than GridSearchCV, the solution quality is not impacted. My Optuna tuned model performed very similarly to the GridSearchCV tuned model in much less time.